# Validation des améliorations de normalisation — POC Data

Objectif :

- valider les améliorations identifiées lors de l'audit ;
- vérifier la récupération des nouveaux champs ;
- mesurer leur complétude ;
- préparer le benchmark étendu.

## Imports

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT / "src"))

from collectionlens.api.open_library_api import search_book_by_isbn
from collectionlens.benchmark.api_benchmark import build_source_table

## Listes ISBN

In [2]:
isbns = [
    "9782351423554",
    "9791042019396",
    "9782820354488",
    "9782290042298",
    "9782845808331",
    "9782413042341",
    "9782384967421",
    "9791043301087",
    "9782374123035",
    "9782374126173",
    "9791039143431",
    "9791026854920",
    "9791039140652",
    "9791026856283",
    "9791039147156",
    "9782203001190",
    "9782858150045",
    "9791038206229",
    "9782822244787",
    "9791038209657",
]

## Validation des améliorations OpenLibrary

In [3]:
df_openlibrary_v2 = build_source_table(
    source_name="openlibrary",
    search_func=search_book_by_isbn,
    isbns=isbns,
)

### Vérification des nouveaux champs

In [4]:
new_fields = [
    "subjects",
    "subject_people",
    "subject_places",
    "subject_times",
    "cover_small",
    "cover_medium",
    "cover_large",
]

available_fields = [
    field
    for field in new_fields
    if field in df_openlibrary_v2.columns
]

df_openlibrary_v2[available_fields].head()

,subjects,subject_people,subject_places,subject_times,cover_small,cover_medium,cover_large
0,"[franchise:ヴィンランド・サガ, series:ヴィンランド・サガ, form:m...","[Lief Ericson, Thorfinn, Thors]",[Japan],[1002-1013],https://covers.openlibrary.org/b/id/14839285-S...,https://covers.openlibrary.org/b/id/14839285-M...,https://covers.openlibrary.org/b/id/14839285-L...
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,[],[],[],[],https://covers.openlibrary.org/b/id/3111095-S.jpg,https://covers.openlibrary.org/b/id/3111095-M.jpg,https://covers.openlibrary.org/b/id/3111095-L.jpg
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Completude de nouveaux champs

In [5]:
new_fields_completeness = (
    df_openlibrary_v2[available_fields]
    .notna()
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

new_fields_completeness.columns = [
    "field",
    "completion_rate",
]

new_fields_completeness

,field,completion_rate
0,subjects,20.0
1,subject_people,20.0
2,subject_places,20.0
3,subject_times,20.0
4,cover_small,15.0
5,cover_medium,15.0
6,cover_large,15.0


### Exemples qualitatifs

In [6]:
qualitative_fields = [
    "isbn_query",
    "title",
    "subjects",
    "subject_people",
    "subject_places",
    "subject_times",
]

available_qualitative_fields = [
    field
    for field in qualitative_fields
    if field in df_openlibrary_v2.columns
]

df_openlibrary_v2[
    df_openlibrary_v2["found"] == True
][available_qualitative_fields]

,isbn_query,title,subjects,subject_people,subject_places,subject_times
0,9782351423554,Vinland Saga (1),"[franchise:ヴィンランド・サガ, series:ヴィンランド・サガ, form:m...","[Lief Ericson, Thorfinn, Thors]",[Japan],[1002-1013]
3,9782290042298,"Fly, tome 1",[],[],[],[]
9,9782374126173,"Sinbad, Tome 1",[Manga],[],[],[]
15,9782203001190,Tintin au Tibet,"[series:Les Aventures de Tintin, Fiction, Comi...","[Tintin (Fictitious character), Tintín (Person...","[France, Belgium, Tibet (China), Nepal, Bélgic...",[]


### Conclusion

L'intégration des nouveaux champs identifiés lors de l'audit permet désormais de récupérer des métadonnées sémantiques absentes de la première version de la normalisation.

Les champs `subjects`, `subject_people`, `subject_places` et `subject_times` présentent un taux de complétude de 20 % sur l'échantillon testé. Les champs de couverture (`cover_small`, `cover_medium`, `cover_large`) atteignent quant à eux 15 %.

Bien que ces taux restent limités, les informations récupérées sont particulièrement riches lorsque les notices sont disponibles. Les sujets, personnages, lieux et périodes constituent des données potentiellement très utiles pour les futurs systèmes de recommandation et le pipeline RAG.

Cette validation confirme qu'OpenLibrary ne doit pas être considéré comme une source principale de couverture ISBN, mais plutôt comme une source d'enrichissement sémantique complémentaire.

## Validation des améliorations BNF

### Vérification des nouveaux champs

In [7]:
from collectionlens.api.bnf_api import search_book_by_isbn

df_bnf_v2 = build_source_table(
    source_name="bnf",
    search_func=search_book_by_isbn,
    isbns=isbns,
)

df_bnf_v2[
    df_bnf_v2["found"]
][
    [
        "isbn_query",
        "title",
        "format",
        "page_count",
        "bnf_ark",
    ]
]

,isbn_query,title,format,page_count,bnf_ark
0,9782351423554,Vinland saga. 1 / Makoto Yukimura,"1 vol. (213 p.) : ill. en noir et en coul., co...",213.0,http://catalogue.bnf.fr/ark:/12148/cb41404031j
2,9782820354488,Blue Exorcist. 32,18 cm,NaN,http://catalogue.bnf.fr/ark:/12148/cb48763865j
4,9782845808331,"Dragon quest : la quête de Daï. 1 / scénario, ...","1 vol. (199 p.) : ill., couv. ill., couv. ill....",199.0,http://catalogue.bnf.fr/ark:/12148/cb410390294
5,9782413042341,"Dragon quest, the adventure of Dai. 1, Les dis...",1 vol. (326 p.) : ill. ; 18 cm,326.0,http://catalogue.bnf.fr/ark:/12148/cb47221792v
10,9791039143431,Star Wars : La citadelle hurlante,1 vol. (non paginé [272 p.]) ; 23 cm,272.0,http://catalogue.bnf.fr/ark:/12148/cb487470845
12,9791039140652,La Chose ([Éd. spéciale] exclusivité FNAC) [sc...,1 vol. (non paginé [ca 120] p.) : ill. en coul...,NaN,http://catalogue.bnf.fr/ark:/12148/cb48579742x
16,9782858150045,Titine au bistrot / Lindingre,"1 vol. (46 p.) : ill. en coul., couv. ill. en ...",46.0,http://catalogue.bnf.fr/ark:/12148/cb409749703
17,9791038206229,"Fleurs de pavés / scénario, dessin, couleurs, ...",1 vol. (54 p.) : ill. en coul. ; 30 cm,54.0,http://catalogue.bnf.fr/ark:/12148/cb47413282s
18,9782822244787,"Stranger trucs / Antoine Piers, Novy ; [dessin...",1 vol. (124 p.) : ill. en coul. ; 22 cm,124.0,http://catalogue.bnf.fr/ark:/12148/cb47588925q


### Completude des nouveaux champs

In [8]:
bnf_fields = [
    "bnf_ark",
    "format",
    "page_count",
]

bnf_fields_completeness = (
    df_bnf_v2[bnf_fields]
    .notna()
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

bnf_fields_completeness.columns = [
    "field",
    "completion_rate",
]

bnf_fields_completeness

,field,completion_rate
0,bnf_ark,45.0
1,format,45.0
2,page_count,35.0


### Exemple qualitatifs

In [9]:
qualitative_fields = [
    "isbn_query",
    "title",
    "authors",
    "publisher",
    "published_date",
    "description",
    "bnf_ark",
    "format",
    "page_count"
]

available_qualitative_fields = [
    field
    for field in qualitative_fields
    if field in df_bnf_v2.columns
]

df_bnf_v2[
    df_bnf_v2["found"] == True
][available_qualitative_fields]

,isbn_query,title,authors,publisher,published_date,description,bnf_ark,format,page_count
0,9782351423554,Vinland saga. 1 / Makoto Yukimura,"[Yukimura, Makoto (1976-....). Auteur du texte]",Kurokawa (Paris),2009,Collection : Collection dirigée par Grégoire H...,http://catalogue.bnf.fr/ark:/12148/cb41404031j,"1 vol. (213 p.) : ill. en noir et en coul., co...",213.0
2,9782820354488,Blue Exorcist. 32,"[Katō, Kazue (1980-....)]",Crunchyroll (Paris),2026,Collection : Shônen up !,http://catalogue.bnf.fr/ark:/12148/cb48763865j,18 cm,NaN
4,9782845808331,"Dragon quest : la quête de Daï. 1 / scénario, ...","[Sanjō, Riku (1964-....). Auteur du texte, Ina...",Éd. Tonkam (Paris),2007,Collection : Shonen,http://catalogue.bnf.fr/ark:/12148/cb410390294,"1 vol. (199 p.) : ill., couv. ill., couv. ill....",199.0
5,9782413042341,"Dragon quest, the adventure of Dai. 1, Les dis...","[Sanjō, Riku (1964-....). Auteur du texte]",Delcourt-Tonkam (Paris),2022,Code à barres commercial : EAN 9782413042341,http://catalogue.bnf.fr/ark:/12148/cb47221792v,1 vol. (326 p.) : ill. ; 18 cm,326.0
10,9791039143431,Star Wars : La citadelle hurlante,"[Aaron, Jason (1973-....), Gillen, Kieron (197...",Panini France (Nice),2026,Collection : Marvel poche,http://catalogue.bnf.fr/ark:/12148/cb487470845,1 vol. (non paginé [272 p.]) ; 23 cm,272.0
12,9791039140652,La Chose ([Éd. spéciale] exclusivité FNAC) [sc...,"[Johns, Geoff (1973-....). Auteur du texte, Wi...",Panini comics (Nice),2025,Code à barres commercial : EAN 9791039140652,http://catalogue.bnf.fr/ark:/12148/cb48579742x,1 vol. (non paginé [ca 120] p.) : ill. en coul...,NaN
16,9782858150045,Titine au bistrot / Lindingre,"[Lindingre, Yan (1969-....). Auteur du texte]","Audie-""Fluide glacial"" (Paris)",2007,Code à barres commercial : EAN 9782858150045,http://catalogue.bnf.fr/ark:/12148/cb409749703,"1 vol. (46 p.) : ill. en coul., couv. ill. en ...",46.0
17,9791038206229,"Fleurs de pavés / scénario, dessin, couleurs, ...","[Frécon, Sylvain (1972-....). Auteur du texte]","""Fluide glacial"" (Paris)",2024,Code à barres commercial : EAN 9791038206229,http://catalogue.bnf.fr/ark:/12148/cb47413282s,1 vol. (54 p.) : ill. en coul. ; 30 cm,54.0
18,9782822244787,"Stranger trucs / Antoine Piers, Novy ; [dessin...","[Piers, Antoine. Auteur du texte, Novy (1966-....",Jungle (Paris),2024,Code à barres commercial : EAN 9782822244787,http://catalogue.bnf.fr/ark:/12148/cb47588925q,1 vol. (124 p.) : ill. en coul. ; 22 cm,124.0


### Conclusion


L'intégration des nouveaux champs identifiés lors de l'audit permet désormais de récupérer des métadonnées sémantiques absentes de la première version de la normalisation.

Les champs `subjects`, `subject_people`, `subject_places` et `subject_times` présentent un taux de complétude de 20 % sur l'échantillon testé. Les champs de couverture (`cover_small`, `cover_medium`, `cover_large`) atteignent quant à eux 15 %.

Bien que ces taux restent limités, les informations récupérées sont particulièrement riches lorsque les notices sont disponibles. Les sujets, personnages, lieux et périodes constituent des données potentiellement très utiles pour les futurs systèmes de recommandation et le pipeline RAG.

L'analyse qualitative montre notamment la présence de genres, thèmes, personnages, lieux et périodes historiques directement exploitables pour enrichir les futures fonctionnalités de recommandation.

Cette validation confirme qu'OpenLibrary ne doit pas être considéré comme une source principale de couverture ISBN, mais plutôt comme une source d'enrichissement sémantique complémentaire à forte valeur ajoutée lorsque les notices sont disponibles.

## Validation des améliorations OpenLibrary

### Vérification des nouveaux champs

In [10]:
from collectionlens.api.google_books_api import search_book_by_isbn

df_google_v2 = build_source_table(
    source_name="google_books",
    search_func=search_book_by_isbn,
    isbns=isbns,
)

df_google_v2[
    df_google_v2["found"]
][
    [
        "isbn_query",
        "title",
        "description",
        "text_snippet",
    ]
]

,isbn_query,title,description,text_snippet
0,9782351423554,Vinland Saga,"Depuis qu'Askeladd, un chef de guerre fourbe e...","Vol. 15: Thorfinn et Einar, enfin libres, reto..."
3,9782290042298,Le précepteur du héros,None,None
4,9782845808331,Dragon quest : la quête de Daï,Depuis la fin de la guerre contre le Roi du Ma...,None
5,9782413042341,Dragon Quest Tome 1,Depuis la fin de la guerre contre le Roi du Ma...,Depuis la fin de la guerre contre le Roi du Ma...
10,9791039143431,La citadelle hurlante,None,None
11,9791026854920,Harley Quinn,None,None
15,9782203001190,Tintin au Tibet,Un avion de ligne à bord duquel le jeune Chino...,"Tintin au Tibet, pure histoire d&#39;amitié, s..."
16,9782858150045,Titine au bistrot,"Si, quand on vous dit Titine vous pensez Marti...",None
17,9791038206229,Fleurs de pavés,None,None
18,9782822244787,Stranger Trucs,None,None


### Completudes de nouveaux champs

In [13]:
google_fields = [
    "description",
    "text_snippet",
]

google_fields_completeness = (
    df_google_v2[google_fields]
    .notna()
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

google_fields_completeness.columns = [
    "field",
    "completion_rate",
]

google_fields_completeness

,field,completion_rate
0,description,25.0
1,text_snippet,15.0


### Exemple qualitatifs

In [14]:
qualitative_fields = [
    "isbn_query",
    "title",
    "authors",
    "publisher",
    "published_date",
    "description",
    "text_snippet",
    "categories",
]

available_qualitative_fields = [
    field
    for field in qualitative_fields
    if field in df_google_v2.columns
]

df_google_v2[
    df_google_v2["found"] == True
][available_qualitative_fields]

,isbn_query,title,authors,publisher,published_date,description,text_snippet,categories
0,9782351423554,Vinland Saga,"[Makoto Yukimura, Xavière Daumarie]",None,2009-01-15,"Depuis qu'Askeladd, un chef de guerre fourbe e...","Vol. 15: Thorfinn et Einar, enfin libres, reto...",[]
3,9782290042298,Le précepteur du héros,[Riku Sanjô],None,1999-01-01,None,None,[]
4,9782845808331,Dragon quest : la quête de Daï,"[Riku Sanjô, Kōji Inada, 1964-]",None,2007-01-10,Depuis la fin de la guerre contre le Roi du Ma...,None,[]
5,9782413042341,Dragon Quest Tome 1,[],None,2022-03-02,Depuis la fin de la guerre contre le Roi du Ma...,Depuis la fin de la guerre contre le Roi du Ma...,[]
10,9791039143431,La citadelle hurlante,[],None,2026-03-25,None,None,[]
11,9791026854920,Harley Quinn,[],None,2026-02-13,None,None,[]
15,9782203001190,Tintin au Tibet,[Hergé],Editions Moulinsart,1960-01-01,Un avion de ligne à bord duquel le jeune Chino...,"Tintin au Tibet, pure histoire d&#39;amitié, s...",[Young Adult Nonfiction]
16,9782858150045,Titine au bistrot,[Yan Lindingre],Fluide glacial - Audie,2007,"Si, quand on vous dit Titine vous pensez Marti...",None,[]
17,9791038206229,Fleurs de pavés,[Sylvain Frécon],None,2024-02-07,None,None,[]
18,9782822244787,Stranger Trucs,[],None,2024-11-07,None,None,[]


## Conclusion 

L'ajout du champ `text_snippet` permet de récupérer un extrait textuel complémentaire lorsque celui-ci est disponible dans les résultats Google Books.

Sur l'échantillon testé, le taux de complétude atteint 15 % pour `text_snippet` contre 25 % pour `description`. Les exemples observés montrent que ce champ contient principalement des extraits promotionnels ou des résumés courts de l'ouvrage.

L'utilisation de `text_snippet` comme mécanisme de secours permet néanmoins de conserver une information textuelle lorsque la description complète n'est pas disponible.

L'amélioration est donc conservée, mais son apport reste limité par rapport aux enrichissements réalisés sur OpenLibrary et BNF. Google Books demeure avant tout une source généraliste apportant des métadonnées de base et certaines descriptions détaillées lorsque celles-ci sont disponibles.

## Conclusion de la phase de normalisation

L'audit des données brutes a permis d'identifier puis d'intégrer plusieurs améliorations de normalisation sur les différentes sources étudiées.

Les gains les plus significatifs proviennent d'OpenLibrary avec l'ajout de métadonnées sémantiques (sujets, personnages, lieux et périodes) particulièrement intéressantes pour les futurs systèmes de recommandation et le pipeline RAG.

La BNF a été enrichie avec l'identifiant ARK ainsi qu'une extraction du nombre de pages depuis le champ `format`, renforçant la qualité des métadonnées bibliographiques françaises.

Google Books a été complété avec le champ `text_snippet`, dont l'apport reste plus limité mais qui peut servir de mécanisme de secours lorsque la description complète est absente.

Les normalisations sont désormais suffisamment matures pour lancer un benchmark étendu sur un échantillon plus représentatif de la collection et préparer la future stratégie d'agrégation multi-sources.

## Export

In [11]:
output_dir = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "normalization_validation"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

df_openlibrary_v2.to_csv(
    output_dir / "openlibrary_v2_results.csv",
    index=False,
)

new_fields_completeness.to_csv(
    output_dir / "openlibrary_v2_completeness.csv",
    index=False,
)

In [15]:
df_bnf_v2.to_csv(
    output_dir / "bnf_v2_results.csv",
    index=False,
)

bnf_fields_completeness.to_csv(
    output_dir / "bnf_v2_completeness.csv",
    index=False,
)

In [16]:
df_google_v2.to_csv(
    output_dir / "google_books_v2_results.csv",
    index=False,
)

google_fields_completeness.to_csv(
    output_dir / "google_books_v2_completeness.csv",
    index=False,
)